# 🚆 Analyse des Retards Ferroviaires en IDF

> **Objectif :** Explorer les données de trafic et de ponctualité du réseau Transilien afin d'identifier les facteurs corrélés aux retards des trains — trafic en gare, temporalité, mission et ligne de départ.

**Sources de données :**
- `a.csv` — Données opérationnelles des trains (janvier–février 2020) RER A. Dataset plus exhaustifs non trouvés
- `ponctualite-mensuelle-transilien.csv` — Taux de ponctualité mensuel par ligne (2013–2026, open data IDFM)
- `comptage-voyageurs-trains-transilien.csv` — Comptage de voyageurs par gare (open data SNCF)

**Stack technique :** Python · Pandas · NumPy · Plotly · summarytools

---

## Plan d'analyse
1. [Chargement & exploration des données](#1) RER A
2. [Analyse des retards par train et par gare](#2) RER A
3. [Analyse temporelle des retards](#3) RER A
4. [Corrélation trafic voyageurs ↔ ponctualité](#4) Toutes les lignes
5. [Perspectives : événements sociaux & météo](#5) Toutes les ignes


---
## 1. Chargement & Exploration des données <a id='1'></a>

On commence par importer les bibliothèques nécessaires et charger le jeu de données principal.


In [ ]:
! pip install summarytools


In [ ]:
import pandas as pd
import numpy as np
from summarytools import dfSummary
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Configuration globale Plotly
TEMPLATE = "plotly_white"
COLOR_SEQ = px.colors.qualitative.Set2


In [ ]:
df = pd.read_csv("a.csv", sep=";")
print(f"Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

### Aperçu des données
Chaque ligne correspond à un **passage de train** sur la ligne A, avec des informations temporelles, la mission (identifiant de desserte), les gares de départ, et les métriques de retard (max, moyen, médian) en secondes.


In [ ]:
df.head()

In [ ]:
df.info()

### Résumé statistique complet
On utilise `summarytools.dfSummary` pour un panorama rapide des distributions, valeurs manquantes et cardinalités.


In [ ]:

dfSummary(df, is_collapsible=True)

---
## 2. Analyse des retards par train et par gare <a id='2'></a>


### 2.1 Gares de départ
Le dataset couvre **4 gares de départ** de la ligne L :


In [ ]:
df["nom_gare_depart"].value_counts().to_frame("nb_trains").rename_axis("Gare")

### 2.2 Missions opérées
Chaque **mission** correspond à un sillon ferroviaire identifié par un code à 4 lettres. On dénombre **37 missions distinctes** dans le dataset.


In [ ]:
print(f"{df['mission'].nunique()} missions distinctes :")
print(", ".join(sorted(df["mission"].unique())))

### 2.3 Statut des circulations
La très grande majorité des trains circulait normalement sur la période. Les 2,4 % d'annulations constituent néanmoins un signal à surveiller.


In [ ]:
statut_counts = df["statut"].value_counts().reset_index()
statut_counts.columns = ["Statut", "Nombre"]
statut_counts["Part (%)"] = (statut_counts["Nombre"] / len(df) * 100).round(1)

fig = px.pie(
    statut_counts,
    names="Statut",
    values="Nombre",
    title="Répartition des statuts de circulation",
    color_discrete_sequence=COLOR_SEQ,
    template=TEMPLATE
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.update_layout(title_x=0.5)
fig.show()
print(statut_counts.to_string(index=False))


### 2.4 Retard moyen global
Le retard moyen toutes missions confondues s'élève à **~145 secondes (2,4 min)**. Cette valeur masque une forte hétérogénéité entre semaine et week-end, ainsi que des valeurs extrêmes qui tirent la moyenne vers le haut.


In [ ]:
retard_global = df["retard_moyen_en_secondes"].mean()
retard_semaine = df[df["est_semaine"]==1]["retard_moyen_en_secondes"].mean()
retard_weekend = df[df["est_weekend"]==1]["retard_moyen_en_secondes"].mean()

print(f"Retard moyen global    : {retard_global:.0f}s ({retard_global/60:.1f} min)")
print(f"Retard moyen semaine   : {retard_semaine:.0f}s ({retard_semaine/60:.1f} min)")
print(f"Retard moyen week-end  : {retard_weekend:.0f}s ({retard_weekend/60:.1f} min)")

# Dispersion
print(f"\nMédiane globale        : {df['retard_moyen_en_secondes'].median():.0f}s")
print(f"Retard max observé     : {df['retard_max_en_secondes'].max():.0f}s ({df['retard_max_en_secondes'].max()/60:.1f} min)")


### 2.5 Retard moyen par mission (semaine vs week-end)
L'agrégation par mission révèle des écarts importants. Les missions **UZAR, UZEL, TAXE** sont systématiquement les plus en retard — toutes au départ de **Houilles-Carrières-sur-Seine**.


In [ ]:
result = df.groupby("mission").agg(
    retard_moyen=("retard_moyen_en_secondes", "mean"),
    retard_semaine=("retard_moyen_en_secondes", lambda x: x[df.loc[x.index, "est_semaine"] == 1].mean()),
    retard_weekend=("retard_moyen_en_secondes", lambda x: x[df.loc[x.index, "est_weekend"] == 1].mean())
).round(1).reset_index()

# Convertir en minutes
result["retard_moyen_min"] = (result["retard_moyen"] / 60).round(2)
result["retard_semaine_min"] = (result["retard_semaine"] / 60).round(2)
result["retard_weekend_min"] = (result["retard_weekend"] / 60).round(2)

result_sorted = result.sort_values("retard_moyen_min", ascending=False).head(20)

fig = px.bar(
    result_sorted,
    x="mission",
    y=["retard_semaine_min", "retard_weekend_min"],
    barmode="group",
    title="Top 20 missions — Retard moyen en semaine vs week-end (min)",
    labels={"value": "Retard moyen (min)", "mission": "Mission", "variable": "Période"},
    color_discrete_sequence=COLOR_SEQ,
    template=TEMPLATE
)
fig.update_layout(title_x=0.5)
fig.show()


### 2.6 Top 15 des missions les plus en retard
Sur les 15 missions cumulant le plus de retard, **10 partent de Houilles-Carrières-sur-Seine**, ce qui suggère un effet gare de départ à creuser (capacité, trafic, infrastructure).


In [ ]:
top15 = (
    df.groupby(["mission", "nom_gare_depart"])["retard_moyen_en_secondes"]
      .mean()
      .sort_values(ascending=False)
      .head(15)
      .reset_index()
)
top15["retard_moyen_min"] = (top15["retard_moyen_en_secondes"] / 60).round(2)

fig = px.bar(
    top15,
    x="mission",
    y="retard_moyen_min",
    color="nom_gare_depart",
    title="Top 15 — Missions avec le plus fort retard moyen (en minutes)",
    labels={"retard_moyen_min": "Retard moyen (min)", "mission": "Mission", "nom_gare_depart": "Gare de départ"},
    color_discrete_sequence=COLOR_SEQ,
    template=TEMPLATE
)
fig.update_layout(title_x=0.5)
fig.show()


---
## 3. Analyse temporelle des retards <a id='3'></a>

On étudie la distribution des retards selon différentes granularités temporelles : mois, jour de la semaine et jour du mois.


### 3.1 Retard moyen par mois


In [ ]:
df_delay_month = df.groupby("mois")["retard_moyen_en_secondes"].mean() / 60

fig = px.bar(
    x=df_delay_month.index,
    y=df_delay_month.values,
    labels={"x": "Mois", "y": "Retard moyen (min)"},
    title="Retard moyen par mois",
    text_auto=".2f",
    color=df_delay_month.values,
    color_continuous_scale="OrRd",
    template=TEMPLATE
)
fig.update_layout(title_x=0.5, coloraxis_showscale=False)
fig.update_traces(textposition="outside")
fig.show()


### 3.2 Retard moyen par jour de la semaine
Les **lundis et vendredis** concentrent les retards les plus élevés, probablement liés aux pics de trafic du début et de fin de semaine de travail. À l'inverse, le **week-end** présente des retards nettement inférieurs, cohérent avec un trafic voyageurs plus faible.


In [ ]:
df_delay_week = df.groupby("jour_semaine")["retard_moyen_en_secondes"].mean() / 60

jours_labels = {1: "Lundi", 2: "Mardi", 3: "Mercredi", 4: "Jeudi",
               5: "Vendredi", 6: "Samedi", 7: "Dimanche"}

fig = px.bar(
    x=[jours_labels.get(j, str(j)) for j in df_delay_week.index],
    y=df_delay_week.values,
    labels={"x": "Jour", "y": "Retard moyen (min)"},
    title="Retard moyen par jour de la semaine",
    text_auto=".2f",
    color=df_delay_week.values,
    color_continuous_scale="Blues",
    template=TEMPLATE
)
fig.update_layout(title_x=0.5, coloraxis_showscale=False)
fig.update_traces(textposition="outside")
fig.show()


### 3.3 Retard moyen par jour du mois
Cette vue permet de détecter d'éventuels **pics ponctuels** liés à des événements spécifiques (incidents, perturbations, mouvements sociaux) sur les 31 premiers jours.
On observe des valeurs aberrantes le 13 et 27 du mois.

In [ ]:
df_delay_day = df.groupby("jour")["retard_moyen_en_secondes"].mean() / 60

fig = px.line(
    x=df_delay_day.index,
    y=df_delay_day.values,
    labels={"x": "Jour du mois", "y": "Retard moyen (min)"},
    title="Retard moyen par jour du mois",
    markers=True,
    template=TEMPLATE
)
fig.update_traces(line_color="#E45756", marker=dict(size=6))
fig.update_layout(title_x=0.5)
fig.show()


---
## 4. Corrélation trafic voyageurs ↔ ponctualité <a id='4'></a>

On croise deux datasets complémentaires :
- Le **taux de ponctualité mensuel** par ligne (source IDFM/SNCF, 2013–2026)
- Le **comptage de voyageurs** dans les gares IDF (source SNCF Open Data)

> ⚠️ Les données de comptage ne sont disponibles que sur une période partielle (2019–2025), ce qui limite la fenêtre de corrélation aux années 2013–2018 où les deux séries se chevauchent.

Toutes les lignes sont concernés.


### 4.1 Chargement des données de ponctualité


In [ ]:
df_delay = pd.read_csv("ponctualite-mensuelle-transilien.csv", sep=";")
df_delay['annee'] = df_delay['Date'].str[:4]
print(f"Dataset ponctualité : {df_delay.shape[0]:,} lignes, {df_delay['annee'].nunique()} années ({df_delay['annee'].min()} – {df_delay['annee'].max()})")
df_delay.head()


### 4.2 Évolution du taux de ponctualité annuel moyen


In [ ]:
df_delay_year = df_delay.groupby("annee")["Taux de ponctualité"].mean().to_frame(name="ponctualite").reset_index()
df_delay_year["annee_int"] = df_delay_year["annee"].astype(int)

fig = px.line(
    df_delay_year,
    x="annee",
    y="ponctualite",
    markers=True,
    title="Évolution du taux de ponctualité moyen — Réseau (2013–2026)",
    labels={"annee": "Année", "ponctualite": "Taux de ponctualité (%)"},
    template=TEMPLATE
)
fig.update_traces(line_color="#00B4D8", marker=dict(size=7))
fig.update_layout(title_x=0.5, yaxis_range=[85, 95])
fig.show()


Depuis 2017, on observe une croissance du taux de ponctualité des trains. A partir de 2022 ele niveau de ponctualité devient instable mais reste au-dessus de la période avant 2017.
Cause à déterminer

### 4.3 Chargement des données de trafic voyageurs


In [ ]:
df_traffic = pd.read_csv("comptage-voyageurs-trains-transilien.csv", sep=";")
df_traffic_date = df_traffic.groupby("Date")["Somme de Montants"].sum().to_frame(name="traffic").reset_index()
df_traffic_date["annee"] = df_traffic_date["Date"].str[:4]
print(f"Dataset trafic : {df_traffic_date.shape[0]} jours de mesure disponibles")
df_traffic_date.head()


### 4.4 Évolution du trafic par année — vue interactive
Un **menu déroulant** permet de filtrer par année pour observer les tendances intra-annuelles. La vue par année démontre que le comptage n'est pas constant. Il manque des data points.


In [ ]:
annees = sorted(df_traffic_date["annee"].unique())

fig = px.line(df_traffic_date, x="Date", y="traffic",
              labels={"Date": "Date", "traffic": "Nombre de voyageurs"},
              title="Trafic voyageurs dans les gares Transilien",
              template=TEMPLATE)

fig.update_layout(
    xaxis=dict(ticks="outside", ticklen=10, ticklabelshift=25, ticklabelstandoff=15)
)

buttons = []
for annee in annees:
    df_temp = df_traffic_date[df_traffic_date["annee"] == annee]
    buttons.append(dict(
        label=str(annee),
        method="update",
        args=[{"x": [df_temp["Date"]], "y": [df_temp["traffic"]]},
              {"title": f"Trafic voyageurs — Année {annee}"}]
    ))

fig.update_layout(updatemenus=[dict(
    buttons=buttons, direction="down", showactive=True,
    x=1.2, xanchor="left", y=1.15, yanchor="top"
)])
fig.show()


### 4.5 Croisement trafic ↔ ponctualité (2013–2018)
On joint les deux séries sur la clé `annee`. Les années 2019+ ne disposent pas de données de trafic exploitables dans ce dataset.

> **Résultat :** Sur la période commune, le lien entre volume de trafic et taux de ponctualité est visuellement présent mais non linéaire — d'autres facteurs (travaux, social, météo) entrent en jeu.


In [ ]:
df_traffic_year = df_traffic.groupby("Annee")["Somme de Montants"].sum().to_frame(name="traffic").reset_index()
df_traffic_year.columns = ["annee", "traffic"]
df_traffic_year["annee"] = df_traffic_year["annee"].astype(str)

df_merged = df_delay_year.merge(df_traffic_year, on="annee", how="left")
df_merged["annee_int"] = df_merged["annee"].astype(int)
df_corr = df_merged.dropna(subset=["traffic"])

print("Années avec données trafic :", df_corr["annee"].tolist())
corr_val = df_corr["ponctualite"].corr(df_corr["traffic"])
print(f"Corrélation de Pearson (ponctualité ↔ trafic) : {corr_val:.3f}")
df_corr[["annee", "ponctualite", "traffic"]]


In [ ]:
fig = px.scatter(
    df_corr,
    x="traffic",
    y="ponctualite",
    text="annee",
    size="traffic",
    color="ponctualite",
    color_continuous_scale="RdYlGn",
    title="Trafic voyageurs vs Taux de ponctualité (2013–2018)",
    labels={"traffic": "Volume trafic (voyageurs)", "ponctualite": "Taux de ponctualité (%)"},
    template=TEMPLATE
)
fig.update_traces(textposition="top center", marker=dict(opacity=0.8))
fig.update_layout(title_x=0.5)
fig.show()


---
## 5. Perspectives : événements sociaux & météo <a id='5'></a>

Le trafic en gare n'explique qu'une partie des variations de ponctualité. Deux facteurs supplémentaires sont identifiés comme potentiellement significatifs :

### 5.1 Événements sociaux (grèves, manifestations)
Un enrichissement du dataset avec un calendrier des mouvements sociaux permettrait de tester l'hypothèse :
> *Les jours de grève sur la période 2013–2020 coïncident-ils avec les pics de retard ou d'annulation ?*

**Source suggérée :** Répertoire des perturbations sociales SNCF (disponible sur data.gouv.fr)

### 5.2 Données météorologiques
Les conditions météo (pluie intense, neige, canicule) sont connues pour impacter les retards ferroviaires. L'API Météo-France ou Open-Meteo (historique gratuit) permettrait d'ajouter des variables comme :
- Précipitations (mm/j)
- Température min/max
- Vitesse du vent

**Piste de modélisation :** régression linéaire multivariée ou Random Forest pour quantifier la contribution relative de chaque facteur (trafic, jour, météo, social) sur le retard moyen.

```python
# Exemple de structure cible pour la modélisation
# features = ['trafic_voyageurs', 'jour_semaine', 'est_greve', 'precipitation_mm', 'temperature_max']
# target   = 'retard_moyen_en_minutes'
```


---
## Conclusion & synthèse

| Facteur analysé | Résultat clé |
|---|---|
| **Gare de départ** | Houilles-Carrières-sur-Seine concentre 10/15 missions les plus en retard |
| **Jour de la semaine** | Lundi et vendredi = pics de retard ; week-end = retards plus faibles (-60%) |
| **Trafic voyageurs** | Corrélation observable mais non linéaire sur 2013–2018 |
| **Annulations** | 2,4 % des circulations annulées sur la période |
| **Retard moyen** | 2,4 min global ; 2,7 min en semaine vs 1,1 min le week-end |

**Prochaines étapes proposées :**
- Enrichissement avec données météo et calendrier social
- Extension de la période d'analyse (post-COVID)
- Modélisation prédictive du retard par machine learning

---
*Notebook réalisé dans le cadre d'une démarche d'analyse Data — Réseau Ferroviaire IDF*
